# HyperTools 1.0 dev/testing notebook

Interactive companion to the `dev-1.0` modernization branch (see `notes/hypertools_1.0_roadmap.md`).

**Purpose:** exercise EVERY public function across the full use-case matrix, visually verify output, and compare backends. Each section corresponds to one public API function. Run top-to-bottom in Jupyter, Colab, or Kaggle — the three environments we must support.

**Use-case matrix** (applied to each function where relevant):

| dimension | cases |
|-|-|
| input type | single ndarray, list of ndarrays, DataFrame, list of DataFrames, nested lists (→ MultiIndex), text |
| dimensionality | 2D, 3D, high-D (reduced), 1D edge case |
| missing data | none, NaNs (PPCA fill) |
| styling | fmt strings, hue/group, legend, labels, title, multilevel-index color/thickness/opacity |
| coloring | categorical labels, continuous values, mixture proportions, user matrices, multicolored lines |
| models | reduce (PCA/IncrementalPCA/TSNE/UMAP), cluster (KMeans/HDBSCAN), mixtures (GaussianMixture/BayesianGaussianMixture/LDA/NMF), align (hyper/SRM) |
| animation | static, animate=True, spin, sliding window |
| backend | matplotlib (default), plotly (interactive) — 2.0 feature |

In [ ]:
# Setup — when developing locally, install the branch in editable mode first:
#   pip install -e .
import sys, os, time
import numpy as np
import pandas as pd

t0 = time.time()
import hypertools as hyp
print(f'hypertools {hyp.__version__} imported in {time.time()-t0:.2f}s')  # 1.0 target: < 1s

ENV = 'colab' if 'google.colab' in sys.modules else ('kaggle' if os.path.exists('/kaggle') else 'local')
print('environment:', ENV)

In [ ]:
# Shared synthetic datasets (seeded for reproducibility)
rng = np.random.default_rng(42)
walk = np.cumsum(rng.standard_normal((200, 10)), axis=0)
walk2 = np.cumsum(np.random.default_rng(1).standard_normal((200, 10)), axis=0)
clusters = np.vstack([rng.standard_normal((60, 5)) + 6 * i for i in range(3)])
# OVERLAPPING blobs (1.5 sd apart) for mixture-model demos: points in the
# overlap regions have genuinely mixed memberships -> visible color blending
oclusters = np.vstack([rng.standard_normal((100, 5)) + 1.5 * i for i in range(3)])
labels = [f'group{i}' for i in range(3) for _ in range(60)]
df = pd.DataFrame(walk, columns=[f'f{i}' for i in range(10)])
walk_missing = walk.copy(); walk_missing[rng.random(walk.shape) < 0.05] = np.nan

## 1. `hyp.plot` — core static cases

In [ ]:
geo = hyp.plot(walk)                      # 3D trajectory, single array
hyp.plot([walk, walk2])                   # list -> auto color per element
hyp.plot(clusters, 'o')                   # scatter via fmt string
hyp.plot(walk, '--', ndims=2)             # 2D + dashed linestyle (regression: linestyle parsing)
hyp.plot(df);                             # DataFrame input

## 2. `hyp.plot` — models: reduce / cluster / align

In [ ]:
hyp.plot(clusters, 'o', hue=labels, legend=True)
hyp.plot(clusters, 'o', cluster='KMeans', n_clusters=3)
hyp.plot(clusters, 'o', reduce='TSNE')
hyp.plot([walk, walk + 0.5], align='hyper')
hyp.plot(walk_missing);                   # PPCA missing-data interpolation

## 3. `hyp.plot` — animation

Known-broken on master: numpy>=2 Jupyter animations (#265), Colab `animate=True` (#235), figures in loops (#264). These cells are the acceptance tests for the 1.0 fixes.

In [ ]:
# Animations render inline via to_jshtml (works in any Jupyter frontend,
# including static notebook viewers with JS enabled)
from IPython.display import HTML, Image as IPyImage

geo = hyp.plot(walk, animate=True, duration=4, frame_rate=15, show=False)
display(HTML(geo.line_ani.to_jshtml()))

geo = hyp.plot(walk, animate='spin', duration=4, frame_rate=15, show=False)
display(HTML(geo.line_ani.to_jshtml()))

# regression #264: multiple figures in a loop stay distinct
for seed in range(3):
    hyp.plot(np.cumsum(np.random.default_rng(seed).standard_normal((100, 5)), axis=0))

# export animations: extension picks the format (.gif / .png [APNG] / .mp4)
hyp.plot(walk, animate=True, duration=3, frame_rate=10,
         save_path='animation_demo.gif', show=False)
display(IPyImage('animation_demo.gif'))

## 4. `hyp.plot` — 1.0 features (IMPLEMENTED on dev-1.0)

Backend switching, nested-list multilevel styling, mixture-model soft clustering, and matrix-valued hue. Each of these also has pytest coverage (`test_interactive.py`, `test_nested.py`, `test_colors.py`, `test_cluster.py`) and screenshot-harness cases.

In [ ]:
# --- backend switching (approved policy: plotly only on Colab/Kaggle) ---
hyp.plot(walk, backend='matplotlib')      # explicit classic renderer
fig = hyp.plot(walk, backend='plotly')    # interactive plotly renderer
hyp.plot(walk, backend='auto')            # plotly on Colab/Kaggle, else matplotlib

# --- plotly ANIMATIONS: interactive frames in the notebook, exportable to
# gif / animated png / mp4 exactly like the matplotlib backend ---
geo = hyp.plot(walk, animate=True, duration=3, backend='plotly')  # play button
hyp.plot(walk, animate='spin', duration=3, backend='plotly',
         save_path='plotly_spin_demo.gif', show=False)
from IPython.display import Image as IPyImage
display(IPyImage('plotly_spin_demo.gif'))

# --- nested lists -> multilevel styling (color by outer group; deeper
# leaves render thinner + fainter) ---
hyp.plot([[walk, walk2], [np.cumsum(np.random.default_rng(7).standard_normal((200, 10)), axis=0)]])

# --- mixture models on OVERLAPPING clusters: multi-class membership shows
# as blended (intermediate) colors between components ---
props = hyp.cluster(oclusters, cluster='GaussianMixture', n_clusters=3)
soft = np.mean(props.max(axis=1) < 0.9)
print(f'mixture proportions: {props.shape}; {soft:.0%} of points have genuinely mixed membership')
hyp.plot(oclusters, 'o', cluster='GaussianMixture', n_clusters=3)

# --- robust coloring: matrix-valued hue (proportions, weights, any matrix) ---
hyp.plot(oclusters, 'o', hue=props)
hyp.plot(walk, 'o', hue=np.arange(len(walk), dtype=float))  # continuous hue

# --- MULTICOLORED LINES: continuous/matrix hue + line fmt colors each
# trajectory continuously along its length (both backends) ---
hyp.plot(walk, hue=np.arange(len(walk), dtype=float))
hyp.plot(walk, hue=np.arange(len(walk), dtype=float), backend='plotly')

# --- apply_model: stack -> fit once -> unstack core ---
embedded = hyp.apply_model([walk, walk2], 'PCA', ndims=3)
print('shared embedding:', [e.shape for e in embedded])

# --- retired in 1.0 (raise clear errors now): plot(group=...), plot(model=...),
# reduce(model=/align=/normalize=), align(method=/align=True), cluster(ndims=) ---

## 5. `hyp.reduce`

In [ ]:
print(hyp.reduce(walk, ndims=3).shape)
print(hyp.reduce([walk, walk2], ndims=2)[0].shape)
print(hyp.reduce(walk, reduce='IncrementalPCA', ndims=4).shape)

## 6. `hyp.align`

In [ ]:
aligned = hyp.align([walk, walk + 0.5])
print([a.shape for a in aligned])
aligned_srm = hyp.align([walk, walk + 0.5], align='SRM')
print([a.shape for a in aligned_srm])

## 7. `hyp.cluster`, `hyp.normalize`, `hyp.analyze`, `hyp.describe`

In [ ]:
print(np.unique(hyp.cluster(clusters, n_clusters=3)))
print(hyp.normalize(walk).mean(axis=0).round(3)[:5])
print(hyp.analyze(walk, ndims=3, normalize='within').shape)
hyp.describe(walk);

## 8. `hyp.load` + text input (network required)

In [ ]:
geo = hyp.load('weights_sample')
geo.plot()
hyp.plot(['the quick brown fox', 'jumped over the lazy dog',
          'machine learning is fun', 'high dimensional data'], 'o');

## 9. Screenshot capture (headless verification)

For the systematic screenshot matrix, run:
```bash
python scripts/generate_baseline_screenshots.py
```
and review PNGs under `tests/screenshots/`.